# Smoking is bad

In [ ]:
# Packages are installed via Pixi (please check pixi.toml)
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# TODO: load the data using the pathlib library to create the path to your data
from pathlib import Path

DATA_PATH = Path.cwd() / "data" / "smoking_data.csv"
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
# TODO: what are the dimensions of the data frame?
print(f"{df.shape[0]} rows, {df.shape[1]} columns") #1314 rows, 6 columns

In [ ]:
# TODO: use .describe() on your data frame: what do you notice about the string columns?
df.describe() #string observations do not appear at all; by default, .describe() only summarises numeric columns.

df.describe(include=["object", "string"]) #there are 3 text columns: outcome, smoker, gender, each with exactly 2 values and no missing values

In [ ]:
# TODO: do not forget to check the null values
df.isna().sum() #there is one column with missing values; the column name is salary and contains 700 missing values

In [ ]:
# TODO: prepare your data frame so that you can work with

# It would be beneficial to drop the meaningless column Unamed: 0; also, we should convert outcome and smoker to 0/1, in order to take means; nevertheless, it is beneficial to add age_group, in order to show age as the confounder
# Salary column can be left alone, since it is not needed for this analysis
df = (
    df.drop(columns=["Unnamed: 0"])                       # leftover index
      .assign(
          alive=lambda d: (d["outcome"] == "Alive").astype(int),
          smokes=lambda d: (d["smoker"] == "Yes").astype(int),
          age_group=lambda d: pd.cut(
              d["age"], bins=[17, 34, 54, 64, 74, 120],
              labels=["18-34", "35-54", "55-64", "65-74", "75+"],
          ),
      )
)
df.head()

## Smoking analysis
Let's first check what your cousin did.

In [ ]:
# TODO: Reimplement the small code snippet described in the problem set instructions.
df.groupby(["smokes"]).agg(prob=("alive", np.mean))

You should get the same results.

# Is smoking really good?
Now it is up to you to run the analysis further and check what could be underlying your cousings results.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(data=df, x="smoker", y="age")
plt.title("Age distribution of smokers vs non-smokers")
plt.show()

In [ ]:
rates = df.groupby(["age_group", "smoker"], observed=True)["alive"].mean().reset_index()
sns.barplot(data=rates, x="age_group", y="alive", hue="smoker")
plt.ylabel("Survival rate")
plt.title("Survival rate by age group and smoking status")
plt.show()

## Interpretation:

## Overall, smokers seem to survive more often. But smokers in this sample are younger on average, and older people are much less likely to survive regardless of smoking. Within each age group, smokers survive at the same or a lower rate than non-smokers. Age is a confounder: the apparent benefit of smoking disappears once we compare people of similar age. The cousin’s conclusion is therefore not supported.